# DORA Metrics for Lead Time Only

## Env

**Output:**
- Working Kernel
- Loaded python libraries

In [ ]:
print("Kernel is working.")

In [ ]:
from dash import dcc
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

## Data Ingestion

**Output:** pandas dataframe *df_lead*.

In [ ]:
# Useful reference: <https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html>
raw_file_path = '/home/lnx_workspaces/bpp_projects/bpp_module2_project/doraview/data/json/change_lead_time.json'

# Try reading with explicit encoding and error handling
try:
    df_lead = pd.read_json(raw_file_path, encoding='utf-8', convert_dates=["deployed_at"]).sort_values(by=["commit_time"])
    print("\nDataframe info:")
    print(df_lead.info())
    print("\nFirst 5 rows:")
    print(df_lead.head())
except Exception as e:
    print(f"Error: {str(e)}")

## Data Manipulation

**Output:** Updates to pandas data frame *df_lead* for use by figures.
- New aggregated month datetime column.
- New Moving Average columns for plotting (SMA, EMA, CMA)

In [ ]:
# Add new row and quater columns
df_lead["commit_month"] = df_lead["commit_time"].dt.month
df_lead["deploy_month"] = df_lead["deploy_time"].dt.month

df_lead.sort_values(by=["commit_time"], inplace=True)

df_lead.head(100)

In [ ]:
# Add moving average column
# https://www.geeksforgeeks.org/pandas/how-to-calculate-moving-average-in-a-pandas-dataframe/

# Simple Moving Average
df_lead["SMA"] = df_lead["lead_time_hours"].rolling(window=30).mean()

# Exponential Moving Average
df_lead["EMA"] = df_lead["lead_time_hours"].ewm(span=30, adjust=True).mean()

# Cumulative Moving Average
df_lead["CMA"] = df_lead["lead_time_hours"].expanding().mean()

df_lead.sort_values(by=["commit_time"], inplace=True)

df_lead.head(100)

## Figures

### ChatGPT - Lead Time for Changes (LTFC)

**Best Graph Type:** Line Chart with moving average.

**Why:** Lead time is duration measured from commit → deploy.

**It is inherently a time-series, so a line graph best shows:**

- Whether lead times are improving
- How stable or volatile the engineering process is
- The impact of process changes (e.g., new CI/CD pipeline)

**Optional secondary views:**

- Box and Whisker Plot for showing distribution of lead times (median, IQR, outliers)
- Scatter Plot per change, with trend line
- Histogram to show distribution of durations

These alternatives help illustrate variability, which is important in discussing reliability.

### Styling

['#636EFA', # the plotly blue you can see above
 '#EF553B',
 '#00CC96',
 '#AB63FA',
 '#FFA15A',
 '#19D3F3',
 '#FF6692',
 '#B6E880',
 '#FF97FF',
 '#FECB52']

### Scatter Plot

In [ ]:
# https://plotly.com/python/line-and-scatter/
fig_lead_scatter = px.scatter(
	data_frame=df_lead,
	title="Lead Time for Changes Scatter Plot",	# Label for the figure.
	x="commit_time",							# Column for use on x-axis
	y="lead_time_hours",							# Column for use on y-axis
	color="application_id",					# Column for use on color grouping
	trendline="ols",						# Add a trendline
	)

fig_lead_scatter.update_layout(template="plotly_dark")

fig_lead_scatter.show()

### Simple Moving Average Plot

In [ ]:
fig_lead_line_sma = px.line(
	data_frame=df_lead,
	title="Lead Time for Changes with EMA Line Plot",	# Label for the figure.
	x="commit_time",							# Column for use on x-axis
	y="SMA",							# Column for use on y-axis
	color="application_id",					# Column for use on color grouping
	)

fig_lead_line_sma.update_layout(template="plotly_dark")

fig_lead_line_sma.show()

### Exponential Moving Average Plot

In [ ]:
fig_lead_line_ema = px.line(
	data_frame=df_lead,
	title="Lead Time for Changes with EMA Line Plot",	# Label for the figure.
	x="commit_time",							# Column for use on x-axis
	y="EMA",							# Column for use on y-axis
	color="application_id",					# Column for use on color grouping
	)

fig_lead_line_ema.update_layout(template="plotly_dark")

fig_lead_line_ema.show()

### Cumulative Moving Average Plot

In [ ]:
fig_lead_line_cma = px.line(
	data_frame=df_lead,
	title="Lead Time for Changes with EMA Line Plot",	# Label for the figure.
	x="commit_time",							# Column for use on x-axis
	y="CMA",							# Column for use on y-axis
	color="application_id",					# Column for use on color grouping
	)

# Apply Plotly colour pallet
fig_lead_line_cma.update_layout(template="plotly_dark")

fig_lead_line_cma.show()

### Dual Scatter & MA with "traces"

In [ ]:
# https://stackoverflow.com/questions/74520782/plotly-express-overlay-two-line-graphs
fig_lead_scat_trace = px.scatter(
	data_frame=df_lead,
	title="Lead Time for Changes Scatter Plot",	# Label for the figure.
	x="commit_time",							# Column for use on x-axis
	y="lead_time_hours",							# Column for use on y-axis
	color="application_id",					# Column for use on color grouping
	)


fig_lead_ema_trace = px.line(
	data_frame=df_lead,
	title="Lead Time for Changes with EMA Line Plot",	# Label for the figure.
	x="commit_time",							# Column for use on x-axis
	y="EMA",							# Column for use on y-axis
	color="application_id",					# Column for use on color grouping
	)

fig_lead_scat_trace.add_traces(
	list(fig_lead_ema_trace.select_traces())
)

fig_lead_scat_trace.update_layout(template="plotly_dark")

fig_lead_scat_trace.show()